In [1]:
#import packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from kneed import KneeLocator

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import calinski_harabasz_score

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN

from sklearn.model_selection import GridSearchCV

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

ModuleNotFoundError: No module named 'kneed'

In [ ]:
data = pd.read_csv('CreditCard_dataset.csv')

# Drop customer ID column
data = data.drop(columns=['CUST_ID'])

# Replace null values correctly (no chained assignment)
data['MINIMUM_PAYMENTS'] = data['MINIMUM_PAYMENTS'].fillna(0)

data.head()

In [ ]:
#Check the NaN count per features
null_count = data.isna().sum()
null_count

In [ ]:
#Drops the one CREDIT LIMIT Missing value row
data.dropna(inplace=True)
data.head()


In [ ]:
scaler = StandardScaler()
scale_data = scaler.fit_transform(data)
scale_data = pd.DataFrame(scale_data, columns=data.columns)
scale_data.head()

In [ ]:
#Fit our dataframe into 2 components
pca = PCA(n_components=2)
pca = pca.fit_transform(scale_data)
pca = pd.DataFrame(pca, columns=['pc1', 'pc2'])
pca.index = data.index
pca.head()

In [ ]:
sum_of_squared_distances = []
#Tests k values between 1 through 17 (inclusive)
K = range(1,18)
for k in K:
    km = KMeans(n_clusters=k, n_init=10)
    km = km.fit(pca)
    #Evaluation Metric to find optimal K parameter
    sum_of_squared_distances.append(km.inertia_)

#Uses elbow method to kind optimal K
location = KneeLocator(range(1,18), sum_of_squared_distances, S=1.0, curve="convex", direction="decreasing")
optimal_k = location.elbow

plt.plot(range(1,18),sum_of_squared_distances)
plt.axvline(optimal_k, color="black", linestyle="--")
plt.xlabel('k')
plt.ylabel('Sum_of_squared_distances')
plt.title('Optimal k selection')
plt.show()

In [ ]:
kmeans = KMeans()

def KMeans_silhouette_score(estimator, X):
    labels = estimator.fit_predict(X)
    score = silhouette_score(X, labels)
    return score

param_grid = {
    'n_init' : [1, 3, 5, 10, 15],
    'init': ['k-means++', 'random'],  
    'max_iter': [100, 200, 300]   
}

grid_search = GridSearchCV(estimator=kmeans, param_grid=param_grid, scoring=KMeans_silhouette_score, cv=5)
grid_search.fit(pca) 

best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Hyperparameters:", best_params)
print("Best Silhouette Score:", best_score)

In [ ]:
best_params['n_clusters'] = optimal_k
km = KMeans(**best_params)

km = km.fit(pca)
labels = km.labels_

pca['pred_label'] = labels
sns.scatterplot(pca, x='pc1', y='pc2', hue='pred_label', palette='viridis')

In [ ]:
range_n_clusters = [3, 4, 5, 6, 7]

for n_clusters in range_n_clusters:
    km = KMeans(n_clusters, n_init=10)
    km.fit(pca)

    silhouette_values = silhouette_samples(pca, km.labels_)
    silhouette_avg = silhouette_score(pca, km.labels_)    
    db_score = davies_bouldin_score(pca, km.labels_)
    ch_score = calinski_harabasz_score(pca, km.labels_)
    
    # Print clustering evaluation metrics
    print(f"Number of clusters: {n_clusters}")
    print(f"Silhouette Score: {silhouette_avg:.2f}")
    print(f"Davies-Bouldin Index: {db_score:.2f}")
    print(f"Calinski-Harabasz Index: {ch_score:.2f}")
    print("----------------------------------------")

    plt.figure(figsize=(10, 6))
    y_lower = 10
    for i in range(n_clusters):
        ith_cluster_silhouette_values = silhouette_values[km.labels_ == i]
        ith_cluster_silhouette_values.sort()
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = plt.cm.viridis(float(i) / n_clusters)
        plt.fill_betweenx(np.arange(y_lower, y_upper),
                          0, ith_cluster_silhouette_values,
                          facecolor=color, edgecolor=color, alpha=0.7)

        plt.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        y_lower = y_upper + 10
        plt.title(f"Silhouette plot for KMeans clustering (k = {n_clusters})\nAverage silhouette score: {silhouette_avg:.2f}")
    plt.xlabel("Silhouette coefficient values")
    plt.ylabel("Cluster label")
    plt.axvline(x=silhouette_avg, color="red", linestyle="--")
    plt.yticks([])  # Clear the yaxis labels / ticks
    plt.xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
    plt.show()

In [ ]:
df = pd.DataFrame(grid_search.cv_results_)[
    ["param_n_components", "param_covariance_type", "mean_test_score"]
]
df["mean_test_score"] = -df["mean_test_score"]
df = df.rename(
    columns={
        "param_n_components": "Number of components",
        "param_covariance_type": "Type of covariance",
        "mean_test_score": "BIC score",
    }
)
df.sort_values(by="BIC score").head()

In [ ]:
n_components = 2
pca_gmm = PCA(n_components)
pca_gmm = pca_gmm.fit_transform(scale_data.values)

n_components_gmm = 6 
gmm = GaussianMixture(n_components_gmm)
gmm.fit(pca_gmm)

labels = gmm.predict(pca_gmm)

db_score = davies_bouldin_score(pca_gmm, labels)
ch_score = calinski_harabasz_score(pca_gmm, labels)

plt.scatter(pca_gmm[:, 0], pca_gmm[:, 1], c=labels)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', marker='x', s=100)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('GMM Clustering in PCA Space')
plt.colorbar(label='Cluster')
plt.show()

print("Davies-Bouldin Index:", db_score)
print("Calinski-Harabasz Index:", ch_score)

In [2]:
DBSCAN_pca = pd.DataFrame(pca, columns=['pc1', 'pc2'])

dbscan = DBSCAN()
#Creates the silhouette score error function to find the most optimum
def DBSCAN_silhouette_score(estimator, X):
    labels = estimator.fit_predict(X)
    unique_labels = np.unique(labels)
    if len(unique_labels) <= 1 or -1 in unique_labels:
        return -1
    else:
        return silhouette_score(X, labels)
    
param_grid = {
    'eps': [0.1, 0.5, 1.0, 1.5, 2.0],  
    'min_samples': [2, 6, 10, 14, 18, 22]      
}

grid_search = GridSearchCV(estimator=dbscan, param_grid=param_grid, scoring=DBSCAN_silhouette_score, cv=5)

grid_search.fit(DBSCAN_pca)

print("Best Parameters:", grid_search.best_params_)
print("Best Silhouette Score:", grid_search.best_score_)

NameError: name 'pca' is not defined

In [3]:
dbscan = DBSCAN(eps=2, min_samples=4)
dbscan_labels = dbscan.fit_predict(DBSCAN_pca)
dbscan_silhouette_score = silhouette_score(DBSCAN_pca, dbscan_labels)

sns.scatterplot(DBSCAN_pca, x='pc1' ,y='pc2', hue=dbscan_labels)

NameError: name 'DBSCAN' is not defined

In [4]:
km = KMeans(n_clusters=optimal_k, n_init=10)
#Use our scale data to find the clusters
km = km.fit(scale_data)
labels = km.labels_
data['KMeans_pred_label'] = labels
KMeans_stats = data.groupby(by='KMeans_pred_label').mean()
#Creates a dataframe with the summarized stats of each feature and cluster
KMeans_stats.head()

NameError: name 'KMeans' is not defined

In [5]:
plt.figure(figsize=(20, 35))

cluster_order = sorted(data['KMeans_pred_label'].unique())

for i, col in enumerate(data.columns[:-1]):  
    ax = plt.subplot(6, 3, i + 1)
    sns.barplot(data=data, x='KMeans_pred_label', y=col, palette='Set1', order=cluster_order, hue='KMeans_pred_label', width=0.8)
    
    plt.xlabel('Cluster Label')
    plt.ylabel(col)
    plt.title(f'Bivariate Distribution of {col} across Clusters')
    
    ax.get_legend().remove()
    
plt.tight_layout()

cluster_order = sorted(data['KMeans_pred_label'].unique())

NameError: name 'data' is not defined

<Figure size 2000x3500 with 0 Axes>

In [6]:
KMeans_stats = KMeans_stats.transpose()
KMeans_stats

NameError: name 'KMeans_stats' is not defined